In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score


SEED = 42
X, y = make_moons(n_samples=350, noise=0.28, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)


plt.figure(figsize=(7, 5))
sns.scatterplot(x=X_train[:, 0], y=X_train[:, 1], hue=y_train,
                palette='Set1', s=45, alpha=0.8)
plt.title('Training data: each point is one sample')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend(title='Class')
plt.show()


print('训练集样本数：', len(X_train))
print('测试集样本数：', len(X_test))

In [ ]:
"""
### 观察点
- KNN 是一种**基于实例的学习方法**：预测时会用到训练样本。
- `fit()` 主要保存样本与标签；预测时才计算新样本和训练样本之间的距离。
- 这组数据有两个特征，因此可以在二维平面上展示分类区域。
"""

In [4]:
X_simple = [[0], [1], [2], [3]]
y_simple = [0, 0, 1, 1]


knn1 = KNeighborsClassifier(n_neighbors=1)
knn1.fit(X_simple, y_simple)
prediction = knn1.predict([[4]])


distances, indices = knn1.kneighbors([[4]])
print('预测类别：', prediction)
print('最近邻样本：', np.array(X_simple)[indices[0]].ravel())
print('与最近邻的距离：', distances[0])

NameError: name 'KNeighborsClassifier' is not defined

In [ ]:
"""
核心特质一：K 值会改变决策边界

下面把 K 分别设为 1、3、5、15、50。背景颜色代表模型对该区域的预测类别，散点是训练样本。

- **K=1**：边界通常很曲折，对单个样本和噪声敏感。
- **K 较大**：分类区域更平滑，但局部结构可能被忽略。
- K 没有适用于所有数据的固定最佳值，需要通过验证集或交叉验证选择。
"""

In [5]:
def plot_decision_boundary(ax, model, X, y, title):
    # 创建覆盖样本区域的网格，并让模型预测每个网格点
    x_min, x_max = X[:, 0].min() - 0.6, X[:, 0].max() + 0.6
    y_min, y_max = X[:, 1].min() - 0.6, X[:, 1].max() + 0.6
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.22, cmap=plt.cm.coolwarm)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm,
               edgecolors='k', s=20, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')


k_values = [1, 3, 5, 15, 50]
fig, axes = plt.subplots(1, len(k_values), figsize=(19, 4), sharex=True, sharey=True)
for ax, k in zip(axes, k_values):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    plot_decision_boundary(ax, model, X_train, y_train, f'K = {k}')
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [1]:
"""
### 思考
1. 哪张图的边界最曲折？为什么？
2. 当 K 变大时，为什么一些小区域会消失？
3. 如果 K 等于训练样本总数，模型会倾向于怎样预测？


**核心结论：** K 控制局部投票的范围。小 K 方差通常较大、容易过拟合；大 K 决策更平滑，但可能欠拟合。
## 4. 核心特质二：训练准确率和测试准确率的变化


训练集表现好，不代表对新数据也好。下面比较不同 K 的训练集与测试集准确率。
"""

SyntaxError: invalid character in identifier (2220235121.py, line 2)

In [ ]:
k_values = range(1, 61)
train_scores, test_scores = [], []


for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, model.predict(X_train)))
    test_scores.append(accuracy_score(y_test, model.predict(X_test)))


best_index = int(np.argmax(test_scores))
best_k = list(k_values)[best_index]


plt.figure(figsize=(9, 5))
plt.plot(list(k_values), train_scores, label='Train accuracy')
plt.plot(list(k_values), test_scores, label='Test accuracy')
plt.axvline(best_k, linestyle='--', alpha=0.7, label=f'Best test K={best_k}')
plt.xlabel('Number of neighbors (K)')
plt.ylabel('Accuracy')
plt.title('KNN: accuracy versus K')
plt.legend()
plt.grid(alpha=0.25)
plt.show()


print(f'本次测试集上准确率最高的 K：{best_k}')
print(f'对应测试集准确率：{test_scores[best_index]:.3f}')

In [3]:
"""
### 如何读这张图
- K=1 时，训练准确率可能很高，因为每个训练点的最近邻通常就是它自己。
- 测试准确率反映对未参与训练的数据的表现，但单次划分有偶然性。
- 这里用测试集找最佳 K 只是为了教学演示；正式建模应在训练集内部使用验证集或交叉验证选 K，最后只用测试集做一次评估。


## 5. 核心特质三：距离度量会改变“最近”的定义


KNN 不只看 K，也要定义距离。常见选择包括：
- `p=2`：闵可夫斯基距离的特例，即欧氏距离。
- `p=1`：曼哈顿距离。


两种距离对同一批点的邻居排序可能不同，因此决策边界也可能不同。
"""

SyntaxError: invalid character in identifier (792442517.py, line 2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, p, label in zip(axes, [1, 2], ['Manhattan distance (p=1)', 'Euclidean distance (p=2)']):
    model = KNeighborsClassifier(n_neighbors=7, p=p)
    model.fit(X_train, y_train)
    plot_decision_boundary(ax, model, X_train, y_train, label)
plt.tight_layout()
plt.show()


print('提示：不同数据集上，两种距离的差别可能明显，也可能不明显。')

In [ ]:
"""
## 6. 核心特质四：特征缩放非常重要


假设一个特征的取值范围是 0–1，另一个特征是 0–10000。直接计算距离时，数值范围大的特征可能主导距离。


下面人为放大第二个特征，再比较未缩放与标准化后的结果。标准化参数只在训练集上拟合，避免数据泄漏。
"""

In [ ]:
rng = np.random.default_rng(SEED)
X_scaled_demo = X.copy()
X_scaled_demo[:, 1] *= 100
Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X_scaled_demo, y, test_size=0.30, random_state=SEED, stratify=y
)


plain_model = KNeighborsClassifier(n_neighbors=7)
plain_model.fit(Xa_train, ya_train)
plain_acc = accuracy_score(ya_test, plain_model.predict(Xa_test))


scaled_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
scaled_model.fit(Xa_train, ya_train)
scaled_acc = accuracy_score(ya_test, scaled_model.predict(Xa_test))


plt.figure(figsize=(7, 4))
plt.bar(['Without scaling', 'With StandardScaler'], [plain_acc, scaled_acc])
plt.ylim(0, 1)
plt.ylabel('Test accuracy')
plt.title('Effect of feature scaling (K=7)')
plt.grid(axis='y', alpha=0.25)
plt.show()


print(f'未缩放测试准确率：{plain_acc:.3f}')
print(f'标准化测试准确率：{scaled_acc:.3f}')

In [ ]:
"""
**结论：** KNN 基于距离，因此对特征量纲敏感。多数真实任务中，应考虑使用 `StandardScaler` 或 `MinMaxScaler`，并把缩放和分类器放进 `Pipeline`。不过，缩放效果取决于数据，不保证每次都提升准确率。
## 7. 核心特质五：距离加权投票

默认 `weights='uniform'` 时，K 个邻居各投一票；使用 `weights='distance'` 时，距离更近的邻居通常拥有更大权重。
"""

In [ ]:
weight_models = {
    'Uniform voting': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=11, weights='uniform')),
    'Distance-weighted voting': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=11, weights='distance')),
}

for name, model in weight_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f'{name}: test accuracy = {accuracy_score(y_test, pred):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, model) in zip(axes, weight_models.items()):
    plot_decision_boundary(ax, model, X_train, y_train, name)
plt.tight_layout()
plt.show()

In [ ]:
"""
## 8. KNN 的优缺点总结

| 特质 | 含义 | 实践建议 |
|---|---|---|
| 惰性学习 | 训练阶段主要保存样本，预测阶段再计算距离 | 数据量大时注意预测成本 |
| K 值敏感 | 小 K 边界更复杂，大 K 更平滑 | 用交叉验证选择 K |
| 距离敏感 | 距离定义会影响邻居排序 | 尝试合适的距离度量 |
| 量纲敏感 | 大范围特征可能支配距离 | 通常先做特征缩放 |
| 预测成本随数据增长 | 朴素方法预测时需比较大量训练点 | 大数据可考虑近似近邻或其他算法 |
| 维度灾难 | 高维空间中距离差异可能变得不明显 | 特征选择、降维或换用其他模型 |

## 9. 自测题
1. 为什么 K=1 容易对噪声敏感？
2. 为什么不能只看训练准确率来选择 K？
3. 为什么标准化要先在训练集上 `fit`，再应用到测试集？
4. `weights='distance'` 与默认均匀投票有什么区别？
5. 如果数据集有 100 万条样本，KNN 可能遇到什么性能问题？

### 最后记住
**KNN 的核心是：用距离找到邻居，再依据邻居标签进行投票。** 学习时重点观察 K、距离度量、特征缩放三者如何共同影响分类结果。
"""